# Giải thích cấu trúc mã nguồn Lab 4: Neural Network

Dưới đây là chi tiết về các thành phần cốt lõi được xây dựng trong chương trình:

### 1. Các hàm kích hoạt (Activation Function)
* **`sigmoid(x)`**: Hàm kích hoạt Sigmoid giúp ánh xạ các giá trị đầu vào thành giá trị trong khoảng $(0, 1)$, biểu diễn xác suất ở đầu ra.
* **`sigmoid_derivative(x)`**: Đạo hàm của hàm sigmoid theo giá trị đầu ra $a = \text{sigmoid}(z)$ (được tính bằng công thức $a(1-a)$), dùng trong quá trình lan truyền ngược (backpropagation).

### 2. Lớp mạng Neural Network (`NeuralNetwork`)
* **Khởi tạo (`__init__`)**: Khởi tạo cấu trúc các tầng (ví dụ `[2, 2, 1]` tương ứng với 2 đầu vào, 1 tầng ẩn 2 node, và 1 đầu ra) cùng với phân phối ngẫu nhiên cho trọng số $W$ và khởi tạo bias $b = 0$.
* **Lan truyền tiến (`fit_partial` - feedforward)**: Tính toán đầu ra của từng tầng bằng cách nhân ma trận trọng số, cộng bias và áp dụng hàm kích hoạt Sigmoid.
* **Lan truyền ngược (`fit_partial` - backpropagation)**: Tính toán gradient của hàm mất mát (Loss) dựa trên Binary Cross-Entropy đối với các trọng số $W$ và bias $b$ từ tầng cuối cùng ngược về tầng đầu tiên.
* **Cập nhật trọng số**: Sử dụng thuật toán Gradient Descent để tối ưu hóa $W$ và $b$ với hệ số học tập `alpha`.
* **Huấn luyện (`fit`)**: Lặp qua số lượng `epochs` chỉ định, thực hiện lan truyền tiến và lùi, đồng thời in ra giá trị Loss sau mỗi chu kỳ để theo dõi quá trình hội tụ.
* **Hàm mất mát (`calculate_loss`)**: Sử dụng hàm mất mát Entropy chéo (Binary Cross-Entropy Loss) để đánh giá độ lỗi của mô hình phân loại nhị phân.

In [ ]:
# Lab4
# L4

# Thêm thư viện
import numpy as np
import pandas as pd

# Hàm sigmoid
def sigmoid(x):
        return 1/(1+np.exp(-x))


# Đạo hàm hàm sigmoid
def sigmoid_derivative(x):
        return x*(1-x)


# Lớp neural network
class NeuralNetwork:
    def __init__(self, layers, alpha=0.1):
		# Mô hình layer ví dụ [2,2,1]
      self.layers = layers

      # Hệ số learning rate
      self.alpha = alpha

      # Tham số W, b
      self.W = []
      self.b = []

      # Khởi tạo các tham số ở mỗi layer
      for i in range(0, len(layers)-1):
            w_ = np.random.randn(layers[i], layers[i+1])
            b_ = np.zeros((layers[i+1], 1))
            self.W.append(w_/layers[i])
            self.b.append(b_)


	# Tóm tắt mô hình neural network
    def __repr__(self):
        return "Neural network [{}]".format("-".join(str(l) for l in self.layers))


	# Train mô hình với dữ liệu
    def fit_partial(self, x, y):
        A = [x]

        # quá trình feedforward
        out = A[-1]
        for i in range(0, len(self.layers) - 1):
            out = sigmoid(np.dot(out, self.W[i]) + (self.b[i].T))
            A.append(out)

        # quá trình backpropagation
        y = y.reshape(-1, 1)
        dA = [-(y/A[-1] - (1-y)/(1-A[-1]))]
        dW = []
        db = []
        for i in reversed(range(0, len(self.layers)-1)):
            dw_ = np.dot((A[i]).T, dA[-1] * sigmoid_derivative(A[i+1]))
            db_ = (np.sum(dA[-1] * sigmoid_derivative(A[i+1]), 0)).reshape(-1,1)
            dA_ = np.dot(dA[-1] * sigmoid_derivative(A[i+1]), self.W[i].T)
            dW.append(dw_)
            db.append(db_)
            dA.append(dA_)

        # Đảo ngược dW, db
        dW = dW[::-1]
        db = db[::-1]

		# Gradient descent
        for i in range(0, len(self.layers)-1):
            self.W[i] = self.W[i] - self.alpha * dW[i]
            self.b[i] = self.b[i] - self.alpha * db[i]

    def fit(self, X, y, epochs=20, verbose=10):
        for epoch in range(0, epochs):
            self.fit_partial(X, y)
            if epoch % verbose == 0:
                loss = self.calculate_loss(X, y)
                print("Epoch {}, loss {}".format(epoch, loss))

	# Dự đoán
    def predict(self, X):
        for i in range(0, len(self.layers) - 1):
            X = sigmoid(np.dot(X, self.W[i]) + (self.b[i].T))
        return X

	# Tính loss function
    def calculate_loss(self, X, y):
        y_predict = self.predict(X)
        #return np.sum((y_predict-y)**2)/2
        return -(np.sum(y*np.log(y_predict) + (1-y)*np.log(1-y_predict)))

# Dataset bài 2
data = pd.read_csv('dataset.csv').values
N, d = data.shape
X = data[:, 0:d-1].reshape(-1, d-1)
y = data[:, 2].reshape(-1, 1)

p = NeuralNetwork([X.shape[1], 2, 1], 0.1)
p.fit(X, y, 10000, 100)

## Nhận xét kết quả huấn luyện mô hình

* **Quá trình hội tụ**: Giá trị Loss (sai số) giảm dần qua các Epoch chứng minh quá trình lan truyền ngược (backpropagation) và cập nhật gradient descent hoạt động chính xác.
* **Tốc độ học (Learning Rate - alpha)**: Với `alpha = 0.1`, mô hình hội tụ một cách mượt mà. Nếu Loss giảm quá chậm, có thể cân nhắc tăng nhẹ `alpha`. Ngược lại, nếu Loss biến động không ổn định, nên giảm `alpha`.
* **Độ phức tạp**: Cấu trúc mạng `[X.shape[1], 2, 1]` đủ khả năng phân tách các tập dữ liệu phân tuyến tính hoặc phi tuyến tính dạng đơn giản (như bài toán XOR).